In [ ]:
import os
import numpy as np
import xarray as xr
import zarr

import dask
import matplotlib.pyplot as plt


In [ ]:
file_paths_dict = {
'03':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_03_20260330_025919/samples.zarr',
'05':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_05_20260330_025920/samples.zarr',
'06':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_06_20260330_025920/samples.zarr',
'04_2':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_04_2_20260330_025919/samples.zarr',
'04_3':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_04_3_20260330_032342/samples.zarr',
'04_0':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_04_0_20260330_034815/samples.zarr',
'02':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_02_20260330_025918/samples.zarr',
'01':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_01_20260330_025918/samples.zarr',
'03_2':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_03_2_20260330_022548/samples.zarr',
'06':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_06_20260330_022522/samples.zarr',
'04':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_04_20260330_022450/samples.zarr',
}

In [ ]:
file_paths_dict = {
'04_0':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae/mse_kl_04_0_20260330_034815/samples.zarr',
}

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

def print_rmse(file_paths_dict, epoches=None, variables=None, samples=None):
    all_records = []

    for label, file_path in file_paths_dict.items():
        try:
            ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        except Exception as e:
            print(f"ERROR loading '{label}': {e}")
            continue

        _variables = ds.variable.values if variables is None else ds.variable.values[variables]
        epochs     = ds.epoch.values    if epoches   is None else ds.epoch.values[epoches]
        samples    = ds.sample.values if samples is None else ds.sample.values[samples]

        # Pre-compute readable dates per sample
        sample_dates = {}
        for sample in samples:
            ts = ds.timestamp.sel(sample=sample).values.squeeze().item()
            sample_dates[sample] = pd.Timestamp(ts, unit='s').strftime('%Y-%m-%d %H:%M')

        for epoch in epochs:
            for var in _variables:
                for sample in samples:
                    orig  = ds.original.sel(epoch=epoch, sample=sample, variable=var).squeeze().values
                    recon = ds.reconstruction.sel(epoch=epoch, sample=sample, variable=var).squeeze().values
                    rmse  = np.sqrt(np.mean((orig - recon) ** 2))
                    max_diff  = np.max(np.abs((orig - recon) ))
                    all_records.append({
                        'Epoch'   : epoch,
                        'Variable': str(var),
                        'Sample'  : str(sample),
                        'Date'    : sample_dates[sample],
                        'Label'   : label,
                        'RMSE'    : rmse,
                        'max_diff': max_diff
                    })

    if not all_records:
        print("No data loaded.")
        return

    df = pd.DataFrame(all_records)
    pivot = df.pivot_table(
        index=['Epoch', 'Variable', 'Sample', 'Date'],
        columns='Label',
        values='RMSE'
    )
    pivot.columns.name = None
    print(pivot.to_string(float_format=lambda x: f"{x:.4f}"))

    df = pd.DataFrame(all_records)
    pivot = df.pivot_table(
        index=['Epoch', 'Variable', 'Sample', 'Date'],
        columns='Label',
        values='max_diff'
    )
    pivot.columns.name = None
    print(pivot.to_string(float_format=lambda x: f"{x:.4f}"))

 



In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

def plot_rmse_curves(file_paths_dict, epoches=None, variables=None, samples=None):
    all_records = []

    for label, file_path in file_paths_dict.items():
        try:
            ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        except Exception as e:
            print(f"ERROR loading '{label}': {e}")
            continue

        _variables = ds.variable.values if variables is None else ds.variable.values[variables]
        _epochs    = ds.epoch.values    if epoches   is None else ds.epoch.values[epoches]
        _samples   = ds.sample.values   if samples   is None else ds.sample.values[samples]

        sample_dates = {}
        for sample in _samples:
            ts = ds.timestamp.sel(sample=sample).values.squeeze().item()
            sample_dates[sample] = pd.Timestamp(ts, unit='s').strftime('%Y-%m-%d %H:%M')

        for epoch in _epochs:
            for var in _variables:
                for sample in _samples:
                    orig  = ds.original.sel(epoch=epoch, sample=sample, variable=var).squeeze().values
                    recon = ds.reconstruction.sel(epoch=epoch, sample=sample, variable=var).squeeze().values
                    diff  = orig - recon
                    all_records.append({
                        'Epoch'   : epoch,
                        'Variable': str(var),
                        'Sample'  : str(sample),
                        'Date'    : sample_dates[sample],
                        'Label'   : label,
                        'RMSE'    : np.sqrt(np.mean(diff ** 2)),
                        'max_diff': np.max(np.abs(diff)),
                    })

    if not all_records:
        print("No data loaded.")
        return

    df = pd.DataFrame(all_records)

    # ── Print tables ──────────────────────────────────────────────────
#    for metric in ['RMSE', 'max_diff']:
#        pivot = df.pivot_table(
#            index=['Epoch', 'Variable', 'Sample', 'Date'],
#            columns='Label', values=metric
#        )
#        pivot.columns.name = None
#        print(f"\n{'─'*60}\n{metric}\n{'─'*60}")
#        print(pivot.to_string(float_format=lambda x: f"{x:.4f}"))

    # ── Plot ──────────────────────────────────────────────────────────
    variables_list = df['Variable'].unique()
    samples_list   = df['Sample'].unique()
    labels         = list(file_paths_dict.keys())

    # one colour per label, consistent across subplots
    cmap   = plt.get_cmap('tab10')
    colors = {label: cmap(i % 10) for i, label in enumerate(labels)}

    for var in variables_list:
        for sample in samples_list:
            sub = df[(df['Variable'] == var) & (df['Sample'] == sample)]
            if sub.empty:
                continue

            date_str = sub['Date'].iloc[0]
            fig, axes = plt.subplots(
                1, 2, figsize=(14, 4),
                constrained_layout=True
            )
            fig.suptitle(
                f"Variable: {var}  |  Sample: {sample}  |  {date_str}",
                fontsize=11, fontweight='bold'
            )

            for metric, ax, title in zip(
                ['RMSE', 'max_diff'],
                axes,
                ['RMSE', 'Max absolute difference']
            ):
                for label in labels:
                    grp = sub[sub['Label'] == label].sort_values('Epoch')
                    if grp.empty:
                        continue
                    ax.plot(
                        grp['Epoch'], grp[metric],
                        label=label,
                        color=colors[label],
                        marker='o', markersize=3,
                        linewidth=1.5
                    )

                ax.set_title(title, fontsize=10)
                ax.set_xlabel('Epoch')
                ax.set_ylabel(metric)
                ax.legend(fontsize=8, ncol=2)
                ax.grid(True, alpha=0.3)

            plt.show()

 



In [ ]:
plot_rmse_curves(file_paths_dict)

In [ ]:
print_rmse(file_paths_dict,   samples=[1])